# DE-01 — Enterprise Data Architecture

**Dataset:** `data/loan_data_01.csv`

This notebook explains how enterprise data moves from a source system to business consumption. It is beginner-friendly, while still introducing the design trade-offs expected from intermediate and advanced data engineers.

## Learning objectives

- Explain source → ingestion → storage → processing → serving → consumption.
- Distinguish OLTP and OLAP boundaries and compare warehouse, lake, and lakehouse platforms.
- Apply Bronze/raw, Silver/conformed, and Gold/serving responsibilities.
- Select batch, micro-batch, streaming, or CDC based on business constraints.
- Classify enterprise use cases and document trust boundaries, ownership, and controls.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path.cwd()
if ROOT.name.lower() == "notebooks":
    ROOT = ROOT.parent

DATA_FILE = ROOT / "data" / "loan_data_01.csv"
assert DATA_FILE.exists(), f"Dataset not found: {DATA_FILE}"

raw = pd.read_csv(DATA_FILE)
print(f"Dataset: {DATA_FILE.name}")
print(f"Rows: {len(raw):,} | Columns: {raw.shape[1]}")
print(raw.head(3).to_string(index=False))

## Learning Content

### 1. End-to-end data flow

| Stage | Purpose | Loan example |
|---|---|---|
| Source | Operational record creation | Loan application system |
| Ingestion | Move data without changing business meaning | Read the CSV export |
| Storage | Retain data at an appropriate trust level | Bronze, Silver, and Gold |
| Processing | Validate, standardize, and enrich | Clean categories and derive total income |
| Serving | Publish stable business models | Approval summary by property area |
| Consumption | Support decisions | Dashboard, report, model, or API |

### 2. OLTP and OLAP boundaries

**OLTP** systems prioritize small, concurrent transactions and current operational state. **OLAP** systems prioritize scans, aggregations, history, and reproducible analysis. Analytical pipelines should avoid placing heavy queries on operational sources.

| Platform | Strength | Trade-off |
|---|---|---|
| Warehouse | Strong SQL governance and performance | Less flexible for raw/unstructured data |
| Data lake | Low-cost, flexible raw storage | Requires strong metadata and quality controls |
| Lakehouse | Lake flexibility with table-management features | More platform and operating complexity |

### 3. Medallion responsibilities

- **Bronze/raw:** preserve source fidelity, provenance, and replayability.
- **Silver/conformed:** enforce types, names, keys, quality rules, and deduplication.
- **Gold/serving:** publish business-ready facts, dimensions, and aggregates.

### 4. Processing-mode selection

- **Batch:** bounded data, predictable schedules, cost efficiency.
- **Micro-batch:** short intervals when seconds-to-minutes latency is acceptable.
- **Streaming:** continuous event processing for very low latency.
- **CDC:** capture inserts, updates, and deletes without repeatedly extracting a full source.

In [ ]:
# Bronze: preserve the source records and attach provenance.
bronze = raw.copy()
bronze["source_file"] = DATA_FILE.name
bronze["bronze_loaded_at"] = pd.Timestamp.now(tz="UTC")

# Silver: standardize values and create a reusable business measure.
silver = bronze.copy()
text_columns = ["Loan_ID", "Gender", "Married", "Dependents", "Education",
                "Self_Employed", "Property_Area", "Loan_Status"]
for column in text_columns:
    silver[column] = silver[column].astype("string").str.strip()

numeric_columns = ["ApplicantIncome", "CoapplicantIncome", "LoanAmount",
                   "Loan_Amount_Term", "Credit_History"]
for column in numeric_columns:
    silver[column] = pd.to_numeric(silver[column], errors="coerce")

silver = silver.drop_duplicates(subset=["Loan_ID"], keep="last")
silver["TotalIncome"] = silver["ApplicantIncome"] + silver["CoapplicantIncome"]

# Gold: publish a stable aggregate for reporting.
gold = (
    silver.groupby(["Property_Area", "Loan_Status"], dropna=False)
    .agg(
        application_count=("Loan_ID", "count"),
        average_loan_amount=("LoanAmount", "mean"),
        average_total_income=("TotalIncome", "mean"),
    )
    .reset_index()
)

print("Bronze rows:", len(bronze))
print("Silver rows:", len(silver))
print("Gold rows:", len(gold))
print(gold.round(2).to_string(index=False))

## Hands-on / Demonstration

### Classify three enterprise use cases

A processing mode is selected from measurable requirements, not product popularity.

In [ ]:
use_cases = pd.DataFrame([
    {
        "use_case": "Daily regulatory loan report",
        "latency": "24 hours",
        "change_pattern": "Bounded daily export",
        "recommended_mode": "Batch",
        "serving_platform": "Warehouse",
    },
    {
        "use_case": "Near-real-time fraud screening",
        "latency": "Seconds",
        "change_pattern": "Continuous application events",
        "recommended_mode": "Streaming + CDC",
        "serving_platform": "Lakehouse / operational analytics",
    },
    {
        "use_case": "Branch approval dashboard",
        "latency": "5–15 minutes",
        "change_pattern": "Frequent small changes",
        "recommended_mode": "Micro-batch or CDC",
        "serving_platform": "Warehouse",
    },
])
print(use_cases.to_string(index=False))

### Draw trust boundaries and ownership

A trust boundary marks where data changes owner, sensitivity, validation level, or permitted use. The table below is a text-based architecture diagram that can be translated into a formal drawing.

In [ ]:
trust_boundaries = pd.DataFrame([
    ["Source → Ingestion", "Loan Operations", "Data Engineering", "Restricted", "Read-only extraction; source throttling"],
    ["Bronze → Silver", "Data Engineering", "Data Engineering", "Internal", "Contract, type, completeness, deduplication"],
    ["Silver → Gold", "Data Engineering", "Analytics Owner", "Approved", "Business rules and reconciliation"],
    ["Gold → Consumption", "Analytics Owner", "BI / Risk Users", "Role-based", "Least privilege and approved metrics"],
], columns=["boundary", "producer", "consumer", "trust_level", "control"])
print(trust_boundaries.to_string(index=False))

## Enterprise Control

Architecture is driven by **SLA, volume, velocity, data sensitivity, and operating model—not product popularity**.

Before approving an architecture, record:

1. Required freshness and recovery time.
2. Current and expected data volume.
3. Arrival rate and burst behavior.
4. Data classification and residency.
5. Source-system impact limits.
6. Ownership, support hours, and escalation path.

In [ ]:
architecture_decision = {
    "sla": "Daily by 07:00",
    "volume": f"{len(raw)} rows in this training partition",
    "velocity": "One bounded CSV delivery",
    "sensitivity": "Confidential loan application data",
    "operating_model": "Batch pipeline owned by Data Engineering",
    "decision": "Batch ingestion into a governed PostgreSQL warehouse",
}

required_factors = {"sla", "volume", "velocity", "sensitivity", "operating_model", "decision"}
assert required_factors.issubset(architecture_decision)
assert len(bronze) == len(silver)
assert int(gold["application_count"].sum()) == len(silver)

print("Architecture decision record")
for key, value in architecture_decision.items():
    print(f"- {key}: {value}")
print("\nDE-01 controls passed.")